In [ ]:
import os
import subprocess as sp
import numpy as np
import pandas as pd
import pyemu
import flopy

In [ ]:
ml = flopy.modflow.Modflow.load("freyberg.nam",model_ws="extra_crispy",load_only=["UPW", "BAS6", "DIS"])
# ml.mg.write_gridSpec("grid.spc")
pp_df = pyemu.utils.setup_pilotpoints_grid(ml, use_ibound_zones=True)

In [ ]:
#np.savetxt("zone.dat",np.ones((ml.nrow,ml.ncol),dtype=np.int),fmt="%2d")
np.savetxt("zone.dat",ml.bas6.ibound.array[0],fmt="%2d")

In [ ]:
args = ["grid.spc","pp_00_pp.dat","0.0","zone.dat","f","structure.complex.dat",\
        '',"struct1","o","1.0e+10","1","10",'struct1','o','1.0e+10','1','10'
        ,'struct1','o','1.0e+10','1','10',"ppk2fac_fac.dat","f",\
        "ppk2fac_stdev.ref","reg.dat"]
with open("ppk2fac.in",'w') as f:
    f.write('\n'.join(args))
os.system("ppk2fac.exe < ppk2fac.in >ppk2fac.stdout")

In [ ]:
ok = pyemu.utils.OrdinaryKrigeOrg("structure.complex.dat","pp_00_pp.dat")

In [ ]:
df_interp = ok.calc_factors_grid(ml.sr,maxpts_interp=10)

In [ ]:
ok.to_grid_factors_file("pyemu_factors.dat")

In [ ]:
pyemu_ref = pyemu.utils.fac2real("pp_00_pp.dat","pyemu_factors.dat",out_file=None)

In [ ]:
ppk2fac_ref = pyemu.utils.fac2real("pp_00_pp.dat","ppk2fac_fac.dat",out_file=None)

In [ ]:
diff = np.abs(pyemu_ref - ppk2fac_ref)
diff[ml.bas6.ibound.array[0]==0] = np.NaN
diff[ml.bas6.ibound.array[0]==-1] = np.NaN
diff

In [ ]:
assert np.nanmax(diff) < 1.0e-6